In [1]:
import pandas as pd
import numpy as np

# Load the clean dataset generated at the end of the cleaning phase
df = pd.read_csv(r'C:\Users\varsh\Downloads\Real estate prediction model\data\cleaned\bengaluru_house_data_clean.csv')

# Also reload delivery_year from our time analysis for age calculations
df_raw = pd.read_csv(r'C:\Users\varsh\Downloads\Real estate prediction model\data\raw\Bengaluru_House_Data.csv')
df['delivery_year'] = df_raw.loc[df.index, 'availability'].apply(
    lambda x: int("20" + x.split('-')[0].strip()) if '-' in str(x) else 2018
)

print(f"Initial shape for feature engineering: {df.shape}")
df.head()

Initial shape for feature engineering: (8746, 8)


,area_type,location,total_sqft,bath,balcony,price,bhk,delivery_year
0,Super built-up Area,Devarabeesana Halli,1672.0,3,2,150.0,3,2019
1,Built-up Area,Devarabeesana Halli,1750.0,3,3,149.0,3,2018
2,Super built-up Area,Devarabeesana Halli,1750.0,3,2,150.0,3,2018
3,Super built-up Area,Devarachikkanahalli,1250.0,2,3,44.0,3,2018
4,Super built-up Area,Devarachikkanahalli,1250.0,2,2,40.0,2,2018


In [2]:
# 1. Price Per Square Foot (Target-derived metric for visualization/analysis)
df['price_per_sqft'] = (df['price'] * 100000) / df['total_sqft']

# 2. Total Rooms (BHK + Bathrooms)
df['total_rooms'] = df['bhk'] + df['bath']

print("Sample of room and price per sqft metrics:")
df[['bhk', 'bath', 'total_rooms', 'total_sqft', 'price_per_sqft']].head()

Sample of room and price per sqft metrics:


,bhk,bath,total_rooms,total_sqft,price_per_sqft
0,3,3,6,1672.0,8971.291866
1,3,3,6,1750.0,8514.285714
2,3,3,6,1750.0,8571.428571
3,3,2,5,1250.0,3520.000000
4,2,2,4,1250.0,3200.000000


In [3]:
# 3. House Age (Proxy relative to the 2018 dataset capture baseline)
# Newer/upcoming properties can have negative or zero age, representing pre-launch or brand new builds
df['house_age'] = 2018 - df['delivery_year']

# 4. Luxury Score
# A simple composite score tracking balconies and premium structural feature densities
# More bathrooms relative to BHK + having a balcony increases the amenity luxury score
df['luxury_score'] = df['balcony'] + (df['bath'] / (df['bhk'] + 1))

print("Sample of engineered Age and Luxury Scores:")
df[['delivery_year', 'house_age', 'luxury_score']].head()

Sample of engineered Age and Luxury Scores:


,delivery_year,house_age,luxury_score
0,2019,-1,2.750000
1,2018,0,3.750000
2,2018,0,2.750000
3,2018,0,3.500000
4,2018,0,2.666667


In [4]:
# 5. Distance to City Center Proxy (Inverse Pricing Density Model)
# Step A: Calculate median price per sqft for each location to represent its premium status
location_premium_idx = df.groupby('location')['price_per_sqft'].median()

# Step B: Map it back to the dataframe. Higher index = Closer to premium commercial core/center
df['city_center_proximity_score'] = df['location'].map(location_premium_idx)

# Fill any unexpected gaps with the global median value
df['city_center_proximity_score'] = df['city_center_proximity_score'].fillna(df['city_center_proximity_score'].median())

print("Geographical core proximity scores successfully engineered:")
df[['location', 'city_center_proximity_score']].head()

Geographical core proximity scores successfully engineered:


,location,city_center_proximity_score
0,Devarabeesana Halli,8571.428571
1,Devarabeesana Halli,8571.428571
2,Devarabeesana Halli,8571.428571
3,Devarachikkanahalli,4305.148257
4,Devarachikkanahalli,4305.148257


In [5]:
# Check unique categories before encoding
print("Unique Property/Area Types:")
print(df['area_type'].unique())

# One-hot encode the area_type column
area_dummies = pd.get_dummies(df['area_type'], prefix='area', drop_first=True, dtype=int)

# Concatenate the new binary columns to our main DataFrame
df_encoded = pd.concat([df, area_dummies], axis=1)

# Drop the original text column since it's now encoded
df_encoded = df_encoded.drop(columns=['area_type'])

print(f"\nDataFrame shape after encoding area_type: {df_encoded.shape}")
df_encoded.head()

Unique Property/Area Types:
<StringArray>
['Super built-up Area', 'Built-up Area', 'Plot Area', 'Carpet Area']
Length: 4, dtype: str

DataFrame shape after encoding area_type: (8746, 15)


,location,total_sqft,bath,balcony,price,bhk,delivery_year,price_per_sqft,total_rooms,house_age,luxury_score,city_center_proximity_score,area_Carpet Area,area_Plot Area,area_Super built-up Area
0,Devarabeesana Halli,1672.0,3,2,150.0,3,2019,8971.291866,6,-1,2.750000,8571.428571,0,0,1
1,Devarabeesana Halli,1750.0,3,3,149.0,3,2018,8514.285714,6,0,3.750000,8571.428571,0,0,0
2,Devarabeesana Halli,1750.0,3,2,150.0,3,2018,8571.428571,6,0,2.750000,8571.428571,0,0,1
3,Devarachikkanahalli,1250.0,2,3,44.0,3,2018,3520.000000,5,0,3.500000,4305.148257,0,0,1
4,Devarachikkanahalli,1250.0,2,2,40.0,2,2018,3200.000000,4,0,2.666667,4305.148257,0,0,1


In [6]:
# Generate dummy columns for locations
location_dummies = pd.get_dummies(df_encoded['location'], prefix='loc', dtype=int)

# Drop the 'loc_other' column to act as the baseline comparison column for the ML model
if 'loc_other' in location_dummies.columns:
    location_dummies = location_dummies.drop(columns=['loc_other'])

# Combine location dummy variables into the main workspace
df_encoded = pd.concat([df_encoded, location_dummies], axis=1)

# Drop the original text location column
df_encoded = df_encoded.drop(columns=['location'])

print(f"Final pipeline dataset dimensions after complete encoding actions: {df_encoded.shape}")

Final pipeline dataset dimensions after complete encoding actions: (8746, 775)


In [7]:
# Create a list of target leakage columns to drop from the ML feature set
# We also drop the delivery_year raw variable since we already extracted its information into house_age
leakage_and_raw_cols = ['price_per_sqft', 'delivery_year']

# Safe check and drop
df_selected = df_encoded.drop(columns=[col for col in leakage_and_raw_cols if col in df_encoded.columns])

print(f"Shape after removing target leakage variables: {df_selected.shape}")

Shape after removing target leakage variables: (8746, 773)


In [8]:
# Dropping total_rooms to avoid multicollinearity issues with bhk and bath
redundant_cols = ['total_rooms']

df_selected = df_selected.drop(columns=[col for col in redundant_cols if col in df_selected.columns])

print(f"Shape after removing redundant features: {df_selected.shape}")

Shape after removing redundant features: (8746, 772)


In [9]:
# Drop weak/noise features that add complexity without predictive power
noise_cols = ['balcony']

df_selected = df_selected.drop(columns=[col for col in noise_cols if col in df_selected.columns])

print(f"Final training-ready dataset shape: {df_selected.shape}")

Final training-ready dataset shape: (8746, 771)


In [10]:
# Save to the processed data directory
final_processed_path = ('C:\\Users\\varsh\\Downloads\\Real estate prediction model\\data\\processed\\bengaluru_house_data_engineered.csv')
df_selected.to_csv(final_processed_path, index=False)

print(f"🎉 Success! Feature Engineering & Selection phase is complete.")
print(f"Your final modeling dataset is saved to: {final_processed_path}")

🎉 Success! Feature Engineering & Selection phase is complete.
Your final modeling dataset is saved to: C:\Users\varsh\Downloads\Real estate prediction model\data\processed\bengaluru_house_data_engineered.csv
